In [ ]:
# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================
!pip install langdetect tqdm pandas matplotlib --quiet
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================
import os
import json
import tarfile
import zipfile
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

In [ ]:
# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
# ============================================================
# 3. SET YOUR .tar.gz PATH
# ============================================================
!tar -xzf "/content/drive/MyDrive/Project_02/liputan6_data.tar.gz" -C "/content/sample_data"
tar_path = "/content/drive/MyDrive/Project_02/liputan6_data.tar.gz"
extract_dir = "/content/sample_data"

print("tar exists:", os.path.exists(tar_path))
print("extract dir exists:", os.path.exists(extract_dir))

In [ ]:
# ============================================================
# 4. SET FOLDER PATHS
# ============================================================
folder_train = "/content/sample_data/liputan6_data/canonical/train"
folder_dev   = "/content/sample_data/liputan6_data/canonical/dev"
folder_test  = "/content/sample_data/liputan6_data/canonical/test"

total_folder_train = sum([len(files) for _, _, files in os.walk(folder_train)])
print("Jumlah file folder_train:", total_folder_train)
print("--" * 50)

total_folder_dev = sum([len(files) for _, _, files in os.walk(folder_dev)])
print("Jumlah file folder_dev:", total_folder_dev)
print("--" * 50)

total_folder_test = sum([len(files) for _, _, files in os.walk(folder_test)])
print("Jumlah file folder_test:", total_folder_test)

In [ ]:
# ============================================================
# 5. LIST FILE PATHS PER SPLIT
# ============================================================
train_files = list(Path(folder_train).rglob("*"))
print("Total extracted item in folder train:", len(train_files))
for p in train_files[:5]:
    print(p)
json_files_train = list(Path(folder_train).rglob("*.json"))
print("-" * 50)

test_files = list(Path(folder_test).rglob("*"))
print("Total extracted item in folder test:", len(test_files))
for p in test_files[:5]:
    print(p)
json_files_test = list(Path(folder_test).rglob("*.json"))
print("-" * 50)

dev_files = list(Path(folder_dev).rglob("*"))
print("Total extracted item in folder dev:", len(dev_files))
for p in dev_files[:5]:
    print(p)
json_files_dev = list(Path(folder_dev).rglob("*.json"))

---
## Data Understanding — Script 1: Quick Overview
**Goal:** Fast, presentable understanding of the full dataset aligned with text summarization training.

Covers:
- Schema & field roles
- Dataset size & quality flags (missing, empty)
- Article & summary length stats
- Compression ratio
- Model token limit readiness (BERT / IndoBERT / mT5)
- Language detection sample
- Sample article–summary pairs
---

In [ ]:
# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

# Approximate subword token count (word × 1.3 for BPE tokenizers)
TOKEN_MULT = 1.3

# Model token limits
MODEL_LIMITS = {
    "BERT / IndoBERT" : 512,
    "mT5-small"       : 512,
    "mT5-base"        : 1024,
}

def tok2txt(nested):
    """Gabungkan list token bertingkat menjadi string teks."""
    if not isinstance(nested, list):
        return ""
    return " ".join(" ".join(s) for s in nested if isinstance(s, list))

def quick_stats(lst):
    """Hitung statistik dasar dari sebuah list numerik."""
    if not lst:
        return {}
    s = sorted(lst)
    n = len(s)
    mean = sum(s) / n
    def p(pct): return round(s[min(int(n * pct / 100), n - 1)], 1)
    return {
        "min"    : s[0],
        "max"    : s[-1],
        "mean"   : round(mean, 1),
        "median" : round(s[n // 2], 1),
        "p10"    : p(10), "p25": p(25),
        "p75"    : p(75), "p90": p(90), "p99": p(99),
    }

def iqr_count(lst):
    """Hitung jumlah outlier menggunakan metode IQR."""
    if len(lst) < 4:
        return 0, 0, 0
    s  = sorted(lst)
    n  = len(s)
    q1 = s[int(n * 0.25)]
    q3 = s[int(n * 0.75)]
    iqr = q3 - q1
    lo  = q1 - 1.5 * iqr
    hi  = q3 + 1.5 * iqr
    return sum(1 for v in lst if v < lo or v > hi), round(lo, 1), round(hi, 1)

print("Helper functions siap digunakan.")

In [ ]:
# ============================================================
# 7. SCAN DATASET — STREAMING (hemat RAM, cocok untuk 200k file)
# ============================================================
# Konfigurasi: ubah MAX_SAMPLE_LANG untuk mempercepat deteksi bahasa
MAX_SAMPLE_LANG = 500   # jumlah file yang dicek bahasanya
SAMPLE_PAIRS    = 5     # jumlah contoh artikel-ringkasan yang ditampilkan

# --- semua file dari ketiga split digabung untuk overview ---
all_json_files = json_files_train + json_files_dev + json_files_test
total_files    = len(all_json_files)
print(f"Total file JSON: {total_files:,}")
print(f"  train : {len(json_files_train):,}")
print(f"  dev   : {len(json_files_dev):,}")
print(f"  test  : {len(json_files_test):,}")

# --- inisialisasi counter ---
parse_errors   = 0
missing_art    = 0
missing_summ   = 0
missing_ext    = 0
empty_art      = 0
empty_summ     = 0

art_words_list    = []
summ_words_list   = []
art_sents_list    = []
summ_sents_list   = []
compression_list  = []
token_list        = []
lang_list         = []
sample_pairs      = []

# --- scan ---
for fp in tqdm(all_json_files, desc="Scanning files"):
    try:
        with open(fp, "r", encoding="utf-8") as f:
            r = json.load(f)
    except Exception:
        parse_errors += 1
        continue

    art_raw  = r.get("clean_article")
    summ_raw = r.get("clean_summary")
    ext_raw  = r.get("extractive_summary")

    if art_raw  is None: missing_art  += 1
    if summ_raw is None: missing_summ += 1
    if ext_raw  is None: missing_ext  += 1

    art_text  = tok2txt(art_raw)
    summ_text = tok2txt(summ_raw)

    if art_text.strip()  == "": empty_art  += 1
    if summ_text.strip() == "": empty_summ += 1

    aw  = len(art_text.split())  if art_text  else 0
    sw  = len(summ_text.split()) if summ_text else 0
    as_ = len(art_raw)  if isinstance(art_raw,  list) else 0
    ss_ = len(summ_raw) if isinstance(summ_raw, list) else 0
    at  = int(aw * TOKEN_MULT)

    art_words_list.append(aw)
    summ_words_list.append(sw)
    art_sents_list.append(as_)
    summ_sents_list.append(ss_)
    token_list.append(at)

    if aw > 0 and sw > 0:
        compression_list.append(round(aw / sw, 2))

    # deteksi bahasa (hanya untuk MAX_SAMPLE_LANG file pertama)
    if len(lang_list) < MAX_SAMPLE_LANG and art_text:
        try:
            lang_list.append(detect(art_text[:300]))
        except Exception:
            lang_list.append("unknown")

    # simpan contoh pasangan artikel-ringkasan
    if len(sample_pairs) < SAMPLE_PAIRS and art_text and summ_text:
        sample_pairs.append({
            "id"       : r.get("id", fp.stem),
            "article"  : art_text,
            "summary"  : summ_text,
            "art_w"    : aw,
            "summ_w"   : sw,
            "ratio"    : round(aw / sw, 2) if sw else 0,
            "art_sents": as_,
            "summ_sents": ss_,
        })

print("\nScan selesai.")

In [ ]:
# ============================================================
# 8. SCHEMA & FIELD ROLES
# ============================================================
schema_data = {
    "Field"        : ["id", "url", "clean_article", "clean_summary", "extractive_summary"],
    "Tipe"         : ["int", "str", "list[list[str]]", "list[list[str]]", "list[int]"],
    "Peran"        : [
        "Metadata — tidak digunakan untuk training",
        "Metadata — tidak digunakan untuk training",
        "★ MODEL INPUT  — artikel yang di-tokenisasi per kalimat",
        "★ MODEL TARGET — ringkasan yang di-tokenisasi per kalimat",
        "Label opsional — indeks kalimat ekstraktif"
    ],
    "Digunakan"    : ["Tidak", "Tidak", "Ya", "Ya", "Opsional"],
}
df_schema = pd.DataFrame(schema_data)
print("=" * 70)
print("SKEMA DATA")
print("=" * 70)
print(df_schema.to_string(index=False))
print()
print("Cara preprocessing:")
print("  article_text = ' '.join(' '.join(sent) for sent in record['clean_article'])")
print("  summary_text = ' '.join(' '.join(sent) for sent in record['clean_summary'])")

In [ ]:
# ============================================================
# 9. DATASET OVERVIEW & KUALITAS DATA
# ============================================================
quality_data = {
    "Pemeriksaan"                         : [
        "Total file JSON",
        "Error saat parsing / load",
        "clean_article tidak ada (None)",
        "clean_summary tidak ada (None)",
        "extractive_summary tidak ada (None)",
        "Teks artikel kosong (setelah join)",
        "Teks ringkasan kosong (setelah join)",
    ],
    "Jumlah" : [
        f"{total_files:,}",
        f"{parse_errors:,}",
        f"{missing_art:,}",
        f"{missing_summ:,}",
        f"{missing_ext:,}",
        f"{empty_art:,}",
        f"{empty_summ:,}",
    ],
    "Persentase" : [
        "100%",
        f"{round(parse_errors  / total_files * 100, 3)}%",
        f"{round(missing_art   / total_files * 100, 3)}%",
        f"{round(missing_summ  / total_files * 100, 3)}%",
        f"{round(missing_ext   / total_files * 100, 3)}%",
        f"{round(empty_art     / total_files * 100, 3)}%",
        f"{round(empty_summ    / total_files * 100, 3)}%",
    ],
    "Status" : [
        "—",
        "⚠ periksa" if parse_errors  > 0 else "✓ bersih",
        "⚠ ada nilai hilang" if missing_art  > 0 else "✓ bersih",
        "⚠ ada nilai hilang" if missing_summ > 0 else "✓ bersih",
        "⚠ ada nilai hilang" if missing_ext  > 0 else "✓ bersih",
        "⚠ teks kosong"      if empty_art    > 0 else "✓ bersih",
        "⚠ teks kosong"      if empty_summ   > 0 else "✓ bersih",
    ]
}
df_quality = pd.DataFrame(quality_data)
print("=" * 70)
print("KUALITAS DATA")
print("=" * 70)
print(df_quality.to_string(index=False))

# Distribusi split
print()
print("-" * 50)
print("Distribusi Split:")
split_df = pd.DataFrame({
    "Split" : ["train", "dev", "test"],
    "Jumlah File": [len(json_files_train), len(json_files_dev), len(json_files_test)],
    "Persentase": [
        f"{round(len(json_files_train)/total_files*100,1)}%",
        f"{round(len(json_files_dev)  /total_files*100,1)}%",
        f"{round(len(json_files_test) /total_files*100,1)}%",
    ]
})
print(split_df.to_string(index=False))

In [ ]:
# ============================================================
# 10. STATISTIK PANJANG TEKS
# ============================================================
metrics = [
    ("Jumlah Kata Artikel",    art_words_list),
    ("Jumlah Kalimat Artikel", art_sents_list),
    ("Jumlah Kata Ringkasan",  summ_words_list),
    ("Jumlah Kalimat Ringkasan", summ_sents_list),
    ("Rasio Kompresi",         compression_list),
]

rows = []
for label, lst in metrics:
    s = quick_stats(lst)
    rows.append({
        "Metrik"  : label,
        "Min"     : s["min"],
        "Max"     : s["max"],
        "Rata-rata": s["mean"],
        "Median"  : s["median"],
        "p10"     : s["p10"],
        "p25"     : s["p25"],
        "p75"     : s["p75"],
        "p90"     : s["p90"],
        "p99"     : s["p99"],
    })

df_stats = pd.DataFrame(rows)
print("=" * 70)
print("STATISTIK PANJANG TEKS")
print("=" * 70)
print(df_stats.to_string(index=False))

# Distribusi jumlah kalimat ringkasan
print()
print("-" * 50)
print("Distribusi Jumlah Kalimat Ringkasan:")
summ_sent_dist = Counter(summ_sents_list)
for k in sorted(summ_sent_dist):
    cnt = summ_sent_dist[k]
    bar = "█" * int(cnt / total_files * 30)
    print(f"  {k} kalimat: {bar}  {cnt:,} file ({round(cnt/total_files*100,1)}%)")

In [ ]:
# ============================================================
# 11. KESIAPAN TOKEN MODEL
# ============================================================
print("=" * 70)
print("KESIAPAN TOKEN MODEL")
print(f"(Estimasi token ≈ jumlah kata × {TOKEN_MULT})")
print("=" * 70)

token_stats = quick_stats(token_list)
print(f"  Median token artikel : ~{token_stats['median']}")
print(f"  p90 token artikel    : ~{token_stats['p90']}")
print(f"  p99 token artikel    : ~{token_stats['p99']}")
print(f"  Max token artikel    : ~{token_stats['max']}")
print()

model_rows = []
for model, lim in MODEL_LIMITS.items():
    over = sum(1 for t in token_list if t > lim)
    pct  = round(over / total_files * 100, 2)
    status = "⚠ Perlu truncation" if over > 0 else "✓ Semua muat"
    rekomendasi = "Truncate di ~" + str(int(lim / TOKEN_MULT)) + " kata" if over > 0 else "Tidak perlu truncation"
    model_rows.append({
        "Model"        : model,
        "Max Token"    : lim,
        "File Melebihi": f"{over:,}",
        "Persentase"   : f"{pct}%",
        "Status"       : status,
        "Rekomendasi"  : rekomendasi,
    })

df_tokens = pd.DataFrame(model_rows)
print(df_tokens.to_string(index=False))

In [ ]:
# ============================================================
# 12. DETEKSI OUTLIER (metode IQR)
# ============================================================
print("=" * 70)
print("DETEKSI OUTLIER (Metode IQR)")
print("=" * 70)

outlier_checks = [
    ("Jumlah kata artikel",   art_words_list),
    ("Jumlah kata ringkasan", summ_words_list),
    ("Rasio kompresi",        compression_list),
]
outlier_rows = []
for label, lst in outlier_checks:
    cnt, lo, hi = iqr_count(lst)
    outlier_rows.append({
        "Metrik"         : label,
        "Batas Normal Lo": lo,
        "Batas Normal Hi": hi,
        "Jumlah Outlier" : f"{cnt:,}",
        "Persentase"     : f"{round(cnt / len(lst) * 100, 2)}%" if lst else "N/A",
        "Catatan"        : "⚠ perlu perhatian" if cnt > 0 else "✓ aman"
    })

df_outlier = pd.DataFrame(outlier_rows)
print(df_outlier.to_string(index=False))

In [ ]:
# ============================================================
# 13. DETEKSI BAHASA (sample)
# ============================================================
print("=" * 70)
print(f"DETEKSI BAHASA (dari {len(lang_list)} file pertama)")
print("=" * 70)

lang_counter = Counter(lang_list)
lang_rows = []
for lang, cnt in lang_counter.most_common():
    lang_rows.append({
        "Bahasa"     : lang,
        "Jumlah"     : cnt,
        "Persentase" : f"{round(cnt / len(lang_list) * 100, 1)}%",
    })

df_lang = pd.DataFrame(lang_rows)
print(df_lang.to_string(index=False))

dominant = lang_counter.most_common(1)[0][0] if lang_counter else "unknown"
print(f"\nBahasa dominan: {dominant}")
if dominant == "id":
    print("✓ Bahasa Indonesia terdeteksi — IndoBERT dan mT5 cocok untuk dataset ini.")
else:
    print("⚠ Perlu pengecekan lebih lanjut — bahasa tidak dominan Indonesia.")

In [ ]:
# ============================================================
# 14. CONTOH PASANGAN ARTIKEL–RINGKASAN
# ============================================================
print("=" * 70)
print("CONTOH PASANGAN ARTIKEL – RINGKASAN")
print("=" * 70)

for i, s in enumerate(sample_pairs, 1):
    print(f"\n── Contoh {i} (id: {s['id']} | "
          f"artikel: {s['art_w']} kata, {s['art_sents']} kalimat → "
          f"ringkasan: {s['summ_w']} kata, {s['summ_sents']} kalimat | "
          f"rasio: {s['ratio']}x) ──")
    art_preview  = s["article"][:300] + "..." if len(s["article"]) > 300 else s["article"]
    print(f"ARTIKEL  : {art_preview}")
    print(f"RINGKASAN: {s['summary']}")

In [ ]:
# ============================================================
# 15. VISUALISASI
# ============================================================
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor("#f9f9f7")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

def style_ax(ax, title, xlabel, ylabel="Jumlah File"):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor("#fafaf8")

# 1. Distribusi jumlah kata artikel
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(art_words_list, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
for model, lim in MODEL_LIMITS.items():
    proxy_word = int(lim / TOKEN_MULT)
    ax1.axvline(proxy_word, linestyle="--", linewidth=0.9,
                label=f"{model.split('/')[0].strip()} (~{proxy_word}w)")
ax1.legend(fontsize=7, framealpha=0.7)
style_ax(ax1, "Distribusi Jumlah Kata Artikel", "Kata")

# 2. Distribusi jumlah kata ringkasan
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(summ_words_list, bins=30, color="#55A868", edgecolor="white", linewidth=0.4)
style_ax(ax2, "Distribusi Jumlah Kata Ringkasan", "Kata")

# 3. Distribusi rasio kompresi
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(compression_list, bins=50, color="#C44E52", edgecolor="white", linewidth=0.4)
if compression_list:
    med_cr = sorted(compression_list)[len(compression_list) // 2]
    ax3.axvline(med_cr, color="black", linestyle="--", linewidth=1.2,
                label=f"Median {med_cr:.1f}x")
    ax3.legend(fontsize=8)
style_ax(ax3, "Distribusi Rasio Kompresi", "Rasio (artikel ÷ ringkasan)")

# 4. Distribusi jumlah kalimat artikel
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(art_sents_list, bins=40, color="#8172B2", edgecolor="white", linewidth=0.4)
style_ax(ax4, "Distribusi Jumlah Kalimat Artikel", "Kalimat")

# 5. Distribusi jumlah kalimat ringkasan
ax5 = fig.add_subplot(gs[1, 1])
summ_sent_dist = Counter(summ_sents_list)
ax5.bar(
    [str(k) for k in sorted(summ_sent_dist)],
    [summ_sent_dist[k] for k in sorted(summ_sent_dist)],
    color="#CCB974", edgecolor="white"
)
style_ax(ax5, "Distribusi Jumlah Kalimat Ringkasan", "Jumlah Kalimat")

# 6. Estimasi token artikel vs batas model
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(token_list, bins=60, color="#4C72B0", alpha=0.75, edgecolor="white", linewidth=0.4)
colors_lim = ["#C44E52", "#E8963A", "#55A868"]
for (model, lim), col in zip(MODEL_LIMITS.items(), colors_lim):
    ax6.axvline(lim, color=col, linestyle="--", linewidth=1.0,
                label=f"{model.split('/')[0].strip()} ({lim} tok)")
ax6.legend(fontsize=7, framealpha=0.7)
style_ax(ax6, "Estimasi Token Artikel vs Batas Model", "~Token (kata × 1.3)")

fig.suptitle(
    "Data Understanding — Liputan6 Summarization Dataset\n"
    f"Total: {total_files:,} file | Train: {len(json_files_train):,} | "
    f"Dev: {len(json_files_dev):,} | Test: {len(json_files_test):,}",
    fontsize=13, fontweight="bold", y=1.02
)

plt.savefig("data_understanding_overview.png", bbox_inches="tight",
            dpi=150, facecolor=fig.get_facecolor())
plt.show()
print("Plot disimpan → data_understanding_overview.png")

In [ ]:
# ============================================================
# 16. RINGKASAN AKHIR & CHECKLIST TRAINING
# ============================================================
print("=" * 70)
print("RINGKASAN AKHIR")
print("=" * 70)

summary_rows = [
    ("Total file",               f"{total_files:,}"),
    ("Split train / dev / test", f"{len(json_files_train):,} / {len(json_files_dev):,} / {len(json_files_test):,}"),
    ("Model input",              "clean_article → input_text"),
    ("Model target",             "clean_summary → target_text"),
    ("Median kata artikel",      str(quick_stats(art_words_list)["median"])),
    ("Median kata ringkasan",    str(quick_stats(summ_words_list)["median"])),
    ("Median rasio kompresi",    str(quick_stats(compression_list)["median"])),
    ("File kosong / error",      f"{empty_art + empty_summ + parse_errors:,}"),
    ("Outlier (IQR kata artikel)", str(iqr_count(art_words_list)[0])),
    ("Bahasa dominan",           lang_counter.most_common(1)[0][0] if lang_counter else "unknown"),
]

df_summary = pd.DataFrame(summary_rows, columns=["Metrik", "Nilai"])
print(df_summary.to_string(index=False))

print()
print("=" * 70)
print("CHECKLIST PERSIAPAN TRAINING")
print("=" * 70)
checklist = [
    "1. Join token: ' '.join(token) per kalimat → ' '.join(kalimat) untuk teks penuh",
    "2. Input  → clean_article  (ganti nama kolom menjadi 'input_text')",
    "3. Target → clean_summary  (ganti nama kolom menjadi 'target_text')",
    "4. Hapus baris dengan input_text atau target_text kosong",
    "5. Untuk BERT/IndoBERT (max 512 tok): truncate artikel > ~390 kata",
    "6. Untuk mT5-small (max 512 tok)   : sama dengan poin 5",
    "7. Untuk mT5-base  (max 1024 tok)  : tidak perlu truncation",
    "8. extractive_summary = label bantu opsional, aman untuk diabaikan",
    "9. Split sudah tersedia: train / dev / test — tidak perlu split manual",
]
for item in checklist:
    print(" ", item)